# Building the Taxonomy

#### Imports

In [55]:
%reload_ext autoreload
%autoreload 2

In [56]:
import sys
import os

notebook_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
function_dir = os.path.abspath(os.path.join(parent_dir, 'Function_Files'))
if function_dir not in sys.path:
    sys.path.append(function_dir)

import pandas as pd
from pyvent.tools.llm.openai_api import OpenAIAgent
import datetime
import nest_asyncio
nest_asyncio.apply()

### if one of these fails to load, check what function_dir is - you'll need to make it the directory where the functions are
import Load_Isolate_functions as lif
import Classification_functions as cf

#### Variables - set the category

In [107]:
CATEGORY = "Cutlery"
#Create the input and output path for the data - may need to be edited for whoever's running it
IP_PATH = f"{parent_dir}\\Data\\"
IP_PATH_CAT = f"{parent_dir}\\Data\\{CATEGORY}\\"
OP_PATH = f"{parent_dir}\\Data\\{CATEGORY}\\Output\\"

#### Read in data
sfy = salsify, cat_data = files received from ID for each category

In [ ]:
sfy = pd.read_excel(f"{IP_PATH}All Salsify Items.xlsx", sheet_name='in', skiprows=1)
cat_data = pd.read_excel(f"{IP_PATH_CAT}Cutlery Sales Data for Cost Analysis L6M Feb - July 25.xlsx")
item_master = pd.read_csv(f"{IP_PATH}consolidated_item_master_by_location_20250610152008.csv")

In [62]:
# different ERP Systems may have the same item number for different items - so I am mapping erp system to a number and creating Entity--Item, which is a combined id of ERP System -- Item that uniquely identifies each item

mapping_dict, erp_system_mapped = lif.map_erp_system_to_number(cat_data['ERP System'])

cat_data['Entity--Item'] = cat_data.apply(
    lambda row: (
        f"{mapping_dict.get(str(row['ERP System']).strip(), 0)}--{str(row['Item']).strip().upper()}"
    ),
    axis=1
)

item_master['Entity--Item'] = item_master.apply(
    lambda row: (
        f"{mapping_dict.get(str(row['erp_system_name']).strip(), 0)}--{str(row['item_code']).strip().upper()}"
    ),
    axis=1
)

sfy['Entity--Item'] = '1--'+sfy['S2K Item Number'].astype(str).str.strip().str.upper()

In [63]:
# filter item_master to only items also in the cat
item_master = item_master[item_master['Entity--Item'].isin(cat_data['Entity--Item'])]

In [66]:
### with the help of item_master, we grab PO cost, and fill in any empty Gross Costs or Qty we can
### i.e. if we have PO Cost and Qty but gross cost is missing, we say gross cost = PO Cost × Qty
cat_data_added = lif.add_po_cost(cat_data, item_master)

  Updated 0 transactions with Qty = Net Cost / PO Cost
  Updated 149669 transactions with po_cost_amt = Gross Cost / Qty
PO Cost matching complete:
  Total transactions: 150003
  Matched transactions: 149678
  Match rate: 99.8%
  Total data fixes applied: 149669 (Qty: 0, Gross Cost: 0, Net Cost: 0, PO Cost: 149669)
  Data cleanup complete:
    Removed 321 rows with negative values
    Final row count: 149682 (from 150003)


In [67]:
# group data by Entity--Item
cat_data_grouped = lif.group_data(cat_data_added)

In [108]:
# save grouped data
cat_data_added.to_csv(f"{OP_PATH}Cutlery Data - Cleaned.csv", index=False)

#### Get Relevant Columns for Category

In [71]:
# get columns in salsify with data filled in for the Entity--Items in the category
im_s2k, columns_with_coverage, example_data = lif.get_columns_with_coverage(cat_data_grouped, sfy, 15)

Filtered sfy to 1348 rows
Found 10 columns meeting 15% coverage
Extracted sample data for 10 columns


In [72]:
# some example of data for each column
example_data

,Foodservice Tableware Product Type,Straw & Drink Stirrer Product Type,Product Type Collapse,Pack Size,Color,Material,Material Weight,Product Attributes,Length (IN),Product Dimension Type
0,Fork,Sip Stirrer,Fork,500/Case,White,Polystyrene (PS),Light,WhitePolystyrene (PS)LightN/A,5.000,Standard Product Dimensions
1,Tasting Spoon,Jumbo Straw,Tasting Spoon,3000/Case,White,Polypropylene (PP),Medium,WhitePolypropylene (PP)MediumN/A,7.750,Standard Product Dimensions
2,Fork,Giant Straw,Fork,1000/Case,Red,Plastic,Heavy Duty,RedPlastic,7.750,Standard Product Dimensions
3,Knife,Giant Straw,Sip Stirrer,12/Case,Green,Polylactic Acid (PLA),Medium,GreenPolylactic Acid (PLA)Yes,5.500,Standard Product Dimensions
4,Soup Spoon,Stirrer,Jumbo Straw,10000/Case,White,Polystyrene (PS),Heavyweight,WhitePolystyrene (PS),6.300,Standard Product Dimensions
...,...,...,...,...,...,...,...,...,...,...
95,Knife,Giant Straw,Pick,250/Case,White,Polystyrene (PS),Medium,BlackPolystyrene (PS)Heavy DutyN/A,7.750,Standard Product Dimensions
96,Fork,Straw,Fork,500/Case,Silver,Polypropylene (PP),Heavy Duty,WhitePolypropylene (PP)N/A,10.875,Standard Product Dimensions
97,Knife,Jumbo Straw,Tasting Spoon,1000/Case,Brown,Polylactic Acid (PLA),Medium,MediumFork|Milk Straw|Napkin3,8.000,Standard Product Dimensions
98,Teaspoon,Straw,Fork,7500/Case,Black,Polystyrene (PS),Medium,Polypropylene (PP)MediumFork|Knife|Napkin3,5.750,Standard Product Dimensions


In [73]:
# a list of the columns with coverage
columns_with_coverage

['Foodservice Tableware Product Type',
 'Straw & Drink Stirrer Product Type',
 'Product Type Collapse',
 'Pack Size',
 'Color',
 'Material',
 'Material Weight',
 'Product Attributes',
 'Length (IN)',
 'Product Dimension Type']

In [1]:
# Can manually check/change the columns with coverage to see if they are correct
columns_for_description = ['Foodservice Tableware Product Type',
 'Straw & Drink Stirrer Product Type',
 'Product Type Collapse',
 'Pack Size',
 'Color',
 'Material',
 'Material Weight',
 'Product Attributes',
 'Length (IN)',
 'Product Dimension Type']

#### Merge data with salsify

In [75]:
# get columns with coverage from sfy dataframe and merge with cat_data_grouped dataframe
sfy_covered = sfy[columns_for_description+['Entity--Item']].copy()
cat_data_final = cat_data_grouped.merge(sfy_covered, on='Entity--Item', how='left', suffixes=('', '_dup'))

In [76]:
# check for duplicates - should be none
cat_data_final[cat_data_final.duplicated(subset=['Entity--Item'], keep=False)]
cat_data_final = cat_data_final.drop_duplicates(subset=['Entity--Item'], keep='first')

# fill nan with ''
for col in columns_with_coverage:
    if col in cat_data_final.columns:
        cat_data_final[col] = cat_data_final[col].fillna('')

#### Write files - this is the subsection of to_keep that fits the category

In [ ]:
OP_PATH = f"{parent_dir}\\Data\\{CATEGORY}\\Output\\"
os.makedirs(OP_PATH, exist_ok=True)  
cat_data_final.to_csv(f"{OP_PATH}{CATEGORY}_SKUS_with_Salsify.csv", index=False)

In [78]:
cat_data_final = pd.read_csv(f"{OP_PATH}{CATEGORY}_SKUS_with_Salsify.csv")

## Classification

#### Generate parts for taxonomy prompt

In [87]:
# generate a string of the top 5 values for each column in columns_for_description to help with the prompt
prompt_options_string = cf.get_top_values(cat_data_final, columns_for_description, 5)
output_str = cf.get_most_common_values(prompt_options_string)

In [88]:
# create a combined description string for that will be used in the prompt
cat_data_final['Combined Descriptions'] = (
    cat_data_final['Item Desc 1'].fillna('') + ' ' +
    cat_data_final['Item Desc 2'].fillna('') + ' ' #+
    ).str.strip()

existing_columns_for_description = [col for col in columns_for_description if col in cat_data_final.columns]

if not existing_columns_for_description:
    print("Warning: None of the specified columns for description exist in the DataFrame.")
    cat_data_final['Description with Attributes'] = "" # Create an empty description column
else:
    if len(existing_columns_for_description) < len(columns_for_description):
        missing_cols = set(columns_for_description) - set(existing_columns_for_description)
        print(f"Warning: The following specified columns were not found in the DataFrame and will be skipped: {missing_cols}")
    
cat_data_final['Description with Attributes'] = cat_data_final.apply(
    lambda row: cf.create_description_string(row, existing_columns_for_description),
    axis=1 
)

In [98]:
example_desc, example_output = cf.explain_top_qty_description(cat_data_final, output_str)

Running cost $0.0000:   0%|          | 0/1 [00:00<?, ?chunk/s]

Running cost $0.0000: 100%|██████████| 1/1 [00:01<00:00,  1.25s/chunk]


#### Taxonomy prompts

In [96]:
model = 'gpt-4o-mini' 
chunk_size = 32       
agent = OpenAIAgent(model=model, chunk_size=chunk_size)

In [102]:
# attribute with ai
cat_data_final_tagged = cf.attribute_with_ai(cat_data_final, agent, columns_for_description, prompt_options_string, example_desc, example_output , default_pack_size=1000)

Consider submitting unique prompts to the API to save on costs and time


Running cost $0.5315:   0%|          | 0/148 [00:00<?, ?chunk/s]

Running cost $1.0614: 100%|██████████| 148/148 [06:24<00:00,  2.60s/chunk]


Consider submitting unique prompts to the API to save on costs and time


Running cost $1.3318: 100%|██████████| 148/148 [03:21<00:00,  1.36s/chunk]


#### Write file with just id and taxonomy

In [103]:
# take the ai result and format it properly
# extract_attributes_to_dataframe is just the Entity-Item plus tagged attributes, cat_data_final_tagged_attributed also contains the other variables
cat_data_final_tagged_attributed, extract_attributes_to_dataframe = cf.extract_attributes_to_dataframe(cat_data_final_tagged, columns_for_description, output_excel_filepath=f"{OP_PATH}{CATEGORY}_taxonomy.xlsx", vendor_col = 'VGN')

Processing 4718 rows from the input DataFrame...
Successfully created DataFrame with 4718 rows and 13 columns.
Successfully wrote DataFrame to Excel: c:\Users\zwayne\OneDrive - Advent International\Documents\GitHub\ImperialDadeCategoryManagement\Data\Cutlery\Output\Cutlery_taxonomy.xlsx


In [104]:
# use the pre-ai and post-ai data to see imptovement
cf.coverage_improvement(sfy, cat_data_final, extract_attributes_to_dataframe, columns_for_description)

Computing coverage improvement analysis...
Computed post-LLM coverage for 13 columns
Computed initial coverage for 11 columns
Coverage improvement analysis complete. Found 10 columns to compare.


,% Coverage Post LLM,% Initial Coverage,Difference
Column_Name,,,
Foodservice Tableware Product Type,65.45%,17.40%,48.05%
Straw & Drink Stirrer Product Type,86.90%,4.51%,82.39%
Product Type Collapse,87.18%,25.54%,61.64%
Color,69.54%,15.94%,53.60%
Material,61.21%,15.54%,45.68%
Material Weight,26.79%,7.46%,19.33%
Product Attributes,50.89%,19.27%,31.62%
Length (IN),52.99%,8.37%,44.62%
Product Dimension Type,81.54%,8.41%,73.12%


#### write file with attributes

In [105]:
cat_data_final_tagged_attributed.to_csv(f"{OP_PATH}{CATEGORY}_Attributed.csv", index=False)